[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_00/MFM_ch0_digital_assets/MFM_ch0_digital_assets.ipynb)

# MFM_ch0_digital_assets

**Modelling Financial Markets (MFM) — Chapter 0: Financial Markets in 2026**

Total supply of USD-pegged stablecoins, 2018-2026, and the daily traded value of the iShares Bitcoin Trust (IBIT) spot ETF since its launch in January 2024, together with the Bitcoin price; the distance of Tether (USDT) from its 1 USD peg, 2019-2026, with bands of 0.5% and 1%.

*Author: Daniel Traian Pele*

In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style and data loader

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
Teal     = '#17A2B8'


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def bottom_legend(fig, ncol=3, handles=None, labels=None, fontsize=8):
    """Legend below the figure (outside the axes), after tight_layout; save_fig keeps it (bbox_inches='tight')."""
    plt.tight_layout()
    if handles is None:
        handles, labels, seen = [], [], set()
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in seen and not l.startswith('_'):
                    handles.append(h); labels.append(l); seen.add(l)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False, fontsize=fontsize)


def log_axis(ax, ticks):
    """Log axis with readable labels (no scientific notation)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())


In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../../data/market', '../../data/market', '../data/market', 'data/market']   # cloned repository
START, END = '2000-01-01', '2026-09-18'

# name -> (source, symbol, field)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (EODHD)': ('market', 'EURRON.FOREX', 'close'),
    'EUR/RON (market)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Tether (USDT)': ('market', 'USDT-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'ETF equities': ('fred', 'BOGZ1LM563064100Q', None),
    'All equities': ('fred', 'BOGZ1LM893064105Q', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}
WEEKDAYS_ONLY = {'Gold', 'EUR/USD'}


def read_market(symbol):
    """Daily market series: local copy if the repository is cloned, otherwise GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Single access point to the data: returns a daily pd.Series (no NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # billion USD
    elif source == 'bnr':
        # official BNR rate (RON per unit of currency), yearly XML archives
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


def load(name, start=START, end=END):
    """The series named in SERIES (gold and EUR/USD: weekdays only)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Several series aligned on date (NaN where a market does not trade)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)


## Data

In [ ]:
px = load_panel(['IBIT', 'Bitcoin'])
ibit_volume = load('IBIT volume')
stable = load('Stablecoins')

## Charts

In [ ]:
def fig_stablecoins():
    s = stable[stable > 0].loc['2018':]
    fig, ax = plt.subplots(figsize=(6.6, 2.1))
    ax.fill_between(s.index, s.values, 0, color=Forest, alpha=0.25, lw=0)
    ax.plot(s.index, s.values, color=Forest, lw=0.9)
    ax.set_ylabel('bn USD')
    pts = {}
    peak = s.loc['2022'].idxmax()                         # 2022 peak
    trough = s.loc[peak:'2023-12-31'].idxmin()            # exact trough after the peak
    for tt in [s.loc['2020-01-01':].index[0], peak, trough, s.index[-1]]:
        pts[tt.date()] = s.loc[tt]
        ax.annotate(f'{s.loc[tt]:.1f} bn\n{tt:%d %b %Y}', (tt, s.loc[tt]), xytext=(0, 6),
                    textcoords='offset points', ha='center', fontsize=7)
    ax.set_ylim(0, s.max() * 1.25)
    ax.set_title('Total supply of USD-pegged stablecoins, 2018-2026 (source: DefiLlama)',
                 fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch0_stablecoins')
    return pts

In [ ]:
def fig_usdt_peg():
    """USDT price (close) 2019-2026, deviation from 1 USD in %, bands of +-0.5% and +-1%, March 2020 detail."""
    u = load('Tether (USDT)', start='2019-01-01').dropna()
    dev = 100 * (u - 1)
    fig, ax = plt.subplots(figsize=(6.6, 2.1))
    ax.plot(dev.index, dev.values, color=Forest, lw=0.6, label='USDT price minus 1 USD (%)')
    for b, c, lab in [(0.5, Amber, 'Band of 0.5%'), (1.0, IDAred, 'Band of 1%')]:
        ax.axhline(b, color=c, ls='--', lw=0.8, label=lab)
        ax.axhline(-b, color=c, ls='--', lw=0.8)
    ax.set_ylabel('Deviation (%)'); ax.set_ylim(-3.2, 6.0)
    for t in [dev.idxmin(), dev.idxmax()]:
        ax.annotate(f'{u[t]:.3f} USD\n{t:%d %b %Y}', (t, dev[t]), xytext=(14, 0), textcoords='offset points',
                    fontsize=7, va='center', color='black')
    ins = ax.inset_axes([0.62, 0.52, 0.34, 0.42])
    m = dev.loc['2020-02-24':'2020-04-10']
    ins.plot(m.index, m.values, color=Forest, lw=0.9)
    for b, c in [(0.5, Amber), (1.0, IDAred)]:
        ins.axhline(b, color=c, ls='--', lw=0.6); ins.axhline(-b, color=c, ls='--', lw=0.6)
    ins.set_title('March 2020', fontsize=7, loc='left', pad=2)
    ins.tick_params(labelsize=6); ins.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ins.xaxis.set_major_locator(mdates.DayLocator(bymonthday=[1, 15]))
    ax.set_title('Tether (USDT), 2019-2026: distance from the 1 USD peg', fontsize=9, loc='left')
    bottom_legend(fig, ncol=3)
    save_fig('ch0_usdt_peg')
    return dict(n=len(u), first=u.index[0].date(), last=u.index[-1].date(), within05=(dev.abs() <= 0.5).mean(),
                within1=(dev.abs() <= 1.0).mean(), min=(u.idxmin().date(), u.min()), max=(u.idxmax().date(), u.max()))

In [ ]:
def fig_ibit():
    d = pd.concat([px['IBIT'], ibit_volume, px['Bitcoin']], axis=1, keys=['p', 'v', 'btc']).dropna()
    dv = (d['p'] * d['v'] / 1e9)
    fig, ax = plt.subplots(figsize=(6.6, 2.1))
    ax.bar(dv.index, dv.values, width=1.0, color=MainBlue, alpha=0.35, label='IBIT daily traded value')
    ax.plot(dv.rolling(20).mean().index, dv.rolling(20).mean().values, color=MainBlue, lw=1.0,
            label='20-day average')
    ax.set_ylabel('Traded value (bn USD)')
    ax2 = ax.twinx()
    ax2.plot(d.index, d['btc'] / 1e3, color=Orange, lw=0.9, label='Bitcoin price (k USD, right)')
    ax2.set_ylabel('BTC (thousand USD)', color=Orange)
    ax2.spines['right'].set_visible(True)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.set_title('iShares Bitcoin Trust (IBIT) since launch on 11 Jan 2024: price x volume',
                 fontsize=9, loc='left')
    bottom_legend(fig, ncol=3, handles=h1 + h2, labels=l1 + l2)
    save_fig('ch0_ibit')
    return dict(first=d.index[0].date(), mean_bn=dv.mean(), median_bn=dv.median(), max_bn=dv.max(),
                max_date=dv.idxmax().date(), total_bn=dv.sum())

## Run

In [ ]:
print(fig_stablecoins())
print(fig_usdt_peg())
print(fig_ibit())

![ch0_stablecoins](./ch0_stablecoins.png)

![ch0_usdt_peg](./ch0_usdt_peg.png)

![ch0_ibit](./ch0_ibit.png)

Output: `ch0_stablecoins.pdf`, `ch0_usdt_peg.pdf`, `ch0_ibit.pdf`